# 02 Representation analysis

Trains a small predictive world model (or loads a checkpoint you point to), extracts latents on paired source and tight-clearance trajectories, and runs the analyses used in experiment D: episode-grouped probes, cross-domain decoding, phase-matched CKA with a permutation null, the reuse score, and PCA trajectories.

With the small settings below the numbers are a demonstration of the method, not results.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from contact_state_reuse.analysis import cross_domain_probe, extract_latents, grouped_cv_probe, reuse_score
from contact_state_reuse.analysis.plotting import plot_cka_by_phase, plot_confusion, plot_pca_trajectories
from contact_state_reuse.data import DatasetConfig, generate_dataset, load_dataset
from contact_state_reuse.models import ModelConfig, load_checkpoint
from contact_state_reuse.training import TrainConfig, train_model

root = Path("../data/notebook")
def get(name, domain, n, seed, p):
    path = root / f"{name}.npz"
    if not path.exists():
        generate_dataset(DatasetConfig(name=name, domain=domain, n_episodes=n, seed=seed, p_perturbed=p, out_dir=str(root)))
    return path

train_path = get("rep_train", "source", 80, 0, 0.2)
src_path = get("rep_source", "source", 30, 777, 0.0)
tgt_path = get("rep_tight", "tight_clearance", 30, 777, 0.0)   # same seed -> paired episodes

In [ ]:
CHECKPOINT = None   # set to a path such as outputs/D_reuse_quick/<run>/checkpoints/wm_selfsup_seed0.pt
if CHECKPOINT is None:
    res = train_model([train_path], ModelConfig(kind="world_model", param_budget=40000),
                      TrainConfig(steps=200, log_every=100), "../outputs/notebook", tag="nb_world_model")
    CHECKPOINT = res.checkpoint
model, cfg, norm, _ = load_checkpoint(CHECKPOINT)
src, tgt = load_dataset(src_path), load_dataset(tgt_path)
ls, lt = extract_latents(model, norm, src), extract_latents(model, norm, tgt)
ls.z.shape, lt.z.shape

## Decoding contact mode from frozen latents

Cross-validation folds are grouped by episode, so no episode contributes to both training and testing.

In [ ]:
within = grouped_cv_probe(ls.z, ls.mode, ls.episode)
across = cross_domain_probe(ls.z, ls.mode, lt.z, lt.mode)
print(f"within source: {within.balanced_accuracy:.2f} (chance {within.chance:.2f}); "
      f"train source / test tight clearance: {across.balanced_accuracy:.2f}")
plot_confusion(across.confusion, "../outputs/notebook/confusion_cross.png", "train source, test tight clearance")

## Phase-matched similarity and reuse score

CKA between paired source and shifted latents in each phase, against a null that permutes the episode pairing.

In [ ]:
rr = reuse_score(ls, lt, n_perm=30, n_boot=100)
rows = pd.DataFrame([{"model": cfg.kind, "phase": p, **v} for p, v in rr.per_phase.items()])
print(f"reuse score {rr.score:.3f}, 95% CI {rr.ci[0]:.3f} to {rr.ci[1]:.3f}")
rows

In [ ]:
plot_cka_by_phase(rows, "../outputs/notebook/cka_by_phase.png", "source vs tight clearance")
plot_pca_trajectories(ls, "../outputs/notebook/pca_source.png", title="source")
from IPython.display import Image
Image("../outputs/notebook/pca_source.png")

The PCA view is descriptive. A two-dimensional projection can make continuous variation look clustered, so it is not evidence of discrete states.